In [1]:
import os

FILE_NAMES = {
    # "DAC_NationalDownloadableFile.csv": "cms_doctor_info.csv",
    # "Facility_Affiliation.csv": "cms_facility_info.csv",
    # "MUP_PHY_R25_P05_V20_D23_Prov_Svc.csv": "cms_providers_and_services.csv",
    # "Hospital_General_Information.csv": "cms_hospital_rating.csv",
    # "RBCS_Taxonomy_RY2025.csv": "cms_rbcs_taxonomy.csv",
    # "OP_DTL_GNRL_PGYR2023_P01232026_01102026.csv": "cms_general_payments.csv",
    # "OP_DTL_RSRCH_PGYR2023_P01232026_01102026.csv": "cms_research_payments.csv",
    # "OP_DTL_OWNRSHP_PGYR2023_P01232026_01102026.csv": "cms_ownership_payments.csv",
    # "2026 NPRM ASC Addenda.xlsx": "cms_asc_codes.csv",
    "Hospital_Enrollments_2026.04.01.csv": "cms_facility_info.csv"

}


PATH = r"C:\Users\chika\OneDrive\Desktop\Project\Healthcare\scripts\sql\\"


for file in FILE_NAMES.values():
    row_count = -1 # To avoid counting the header

    with open(os.path.join(PATH, file)) as read_file:
        for line in read_file:
            row_count +=1

    print(f"{file} has {row_count:,} rows pre-SQL load")


cms_facility_info.csv has 9,182 rows pre-SQL load


In [11]:
import pandas as pd
from sqlalchemy import text, create_engine


connection_url = (
    "mssql+pyodbc://Chika\\SQLEXPRESS/BankTransactions?"
    "driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
)
engine = create_engine(connection_url)

# Load the local CSV IDs
csv_df = pd.read_csv(r"C:\Users\chika\OneDrive\Desktop\Project\Healthcare\cms_ownership_payments.csv", usecols=['Physician_NPI'])
csv_ids = set(csv_df['Physician_NPI'])

# Pull the SQL IDs
with engine.connect() as conn:
    sql_ids = pd.read_sql(text("SELECT Physician_NPI FROM bronze.cms_ownership_payments"), conn)
    sql_ids = set(sql_ids['Physician_NPI'])

# Find the missing 132
missing = csv_ids - sql_ids
print(f"Missing Record IDs: {list(missing)[:5]}") # Prints the first 5 missing IDs

DatabaseError: Execution failed on sql 'SELECT Physician_NPI FROM bronze.cms_ownership_payments': (pyodbc.ProgrammingError) ('42S02', "[42S02] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Invalid object name 'bronze.cms_ownership_payments'. (208) (SQLExecDirectW)")
[SQL: SELECT Physician_NPI FROM bronze.cms_ownership_payments]
(Background on this error at: https://sqlalche.me/e/20/f405)

In [12]:
import pandas as pd
from sqlalchemy import create_engine, text

# 1. Connection Setup
connection_url = (
    "mssql+pyodbc://Chika\\SQLEXPRESS/BankTransactions?"
    "driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
)
engine = create_engine(connection_url)

file_path = r"C:\Users\chika\OneDrive\Desktop\Project\Healthcare\cms_ownership_payments.csv"

# 2. Read the CSV
# We load everything as strings first to prevent Pandas from mangling the NPIs
df = pd.read_csv(file_path, dtype=str)

# 3. Data Cleaning (Crucial to match your SQL Table Types)
# Convert money columns to numeric (removes '$' or commas if present)
money_cols = ['Total_Amount_Invested_USDollars', 'Value_of_Interest']
for col in money_cols:
    df[col] = pd.to_numeric(df[col].str.replace('[$,]', '', regex=True), errors='coerce')

# Convert the publication date to a real Python datetime object
df['Payment_Publication_Date'] = pd.to_datetime(df['Payment_Publication_Date'], errors='coerce')

# 4. The Upload
print(f"Attempting to upload {len(df)} rows...")

try:
    # We use schema='bronze' and if_exists='append'
    # index=False is VITAL because you have a Row_ID IDENTITY column in SQL
    df.to_sql(
        name='cms_ownership_payments', 
        con=engine, 
        schema='bronze', 
        if_exists='append', 
        index=False
    )
    
    # 5. Final Verification
    with engine.connect() as conn:
        result = conn.execute(text("SELECT COUNT(*) FROM bronze.cms_ownership_payments"))
        final_count = result.scalar()
        print(f"Success! SQL Row Count: {final_count}")
        
except Exception as e:
    print(f"Upload failed: {e}")

Attempting to upload 4316 rows...
Upload failed: (pyodbc.ProgrammingError) ('42000', '[42000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]The specified schema name "bronze" either does not exist or you do not have permission to use it. (2760) (SQLExecDirectW)')
[SQL: 
CREATE TABLE bronze.cms_ownership_payments (
	[Physician_NPI] VARCHAR(max) NULL, 
	[Recipient_City] VARCHAR(max) NULL, 
	[Recipient_State] VARCHAR(max) NULL, 
	[Physician_Primary_Type] VARCHAR(max) NULL, 
	[Physician_Specialty] VARCHAR(max) NULL, 
	[Total_Amount_Invested_USDollars] FLOAT(53) NULL, 
	[Value_of_Interest] FLOAT(53) NULL, 
	[Terms_of_Interest] VARCHAR(max) NULL, 
	[Submitting_Applicable_Manufacturer_or_Applicable_GPO_Name] VARCHAR(max) NULL, 
	[Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_ID] VARCHAR(max) NULL, 
	[Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_Name] VARCHAR(max) NULL, 
	[Interest_Held_by_Physician_or_an_Immediate_Family_Member] VARCHAR(max) NULL, 
	[Payment_Pub

In [23]:
import pandas as pd
from sqlalchemy import create_engine
import os


KEEP_COLS = {"ownership_payments": [
        "Physician_NPI",
        "Recipient_City",
        "Recipient_State",
        "Physician_Primary_Type",
        "Physician_Specialty",
        "Total_Amount_Invested_USDollars",
        "Value_of_Interest",
        "Terms_of_Interest",
        "Submitting_Applicable_Manufacturer_or_Applicable_GPO_Name",
        "Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_ID",
        "Applicable_Manufacturer_or_Applicable_GPO_Making_Payment_Name",
        "Interest_Held_by_Physician_or_an_Immediate_Family_Member",
        "Payment_Publication_Date"
    ],
}

FILE_NAMES = {
    "OP_DTL_OWNRSHP_PGYR2023_P01232026_01102026.csv": "cms_ownership_payments.csv",
}


PATH = r"C:\Users\chika\OneDrive\Desktop\Project\Healthcare\datasets\\"

def get_engine() -> Engine:
    connection_url = (
        "mssql+pyodbc://Chika\\SQLEXPRESS/NCEH_Clinical_Operations_2026?"
        "driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
    )
    return create_engine(connection_url, fast_executemany=True)


def main():
    engine = get_engine()

    chunk_size = 200000

    try:
        for (input_file, output_file), cols in zip(FILE_NAMES.items(), KEEP_COLS.values()):
            read_file = os.path.join(PATH, input_file)

            print(f"Reading from {input_file}")

            for i, chunk in enumerate(
                pd.read_csv(read_file, chunksize=chunk_size, low_memory=False, usecols=cols)
            ):
                mode = "w" if i == 0 else "a"
                header = True if i == 0 else False

                chunk.to_csv(output_file, index=False, mode=mode, header=header)

                table_name = f"{output_file.replace(".csv", "")}"
                print(f"Loading data into {table_name}: batch {i+1}")
        
                # forgot to add schema="bronze". If you already have a schema, don't forget to add this parameter with the schema name. Otherwise, the table will be automatically saved in a schema known as 'dbo' 
                chunk.to_sql(table_name, engine, schema="bronze", if_exists="append", index=False)

            print(f"Successfully saved data into {output_file}\n ...")
            print(f"Successfully loaded data into {table_name}\n ...")
    except FileNotFoundError as fnfe:
        print(f"Check file name or path again: {fnfe}")
    except Exception as e:
        print(f"An error occured while loading the table: {e}")


if __name__ == "__main__":
    main()

Reading from OP_DTL_OWNRSHP_PGYR2023_P01232026_01102026.csv
Loading data into cms_ownership_payments: batch 1
Successfully saved data into cms_ownership_payments.csv
 ...
Successfully loaded data into cms_ownership_payments
 ...
